---
title: "Cost-per-click surge attribution"
author: "Andratx Bellmunt"
abstract: >
  Adapted from a real business scenario. A +18% cost-per-click surge is observed in aggregated data for 1,900+ digital advertising campaigns. Rate-mix decomposition used to properly attribute the contribution of each individual campaign to the global figure.
format:
  html:
    code-fold: true
    self-contained: true
    include-after-body: _tracker.html
jupyter: python3
number-sections: true
---

# Initialization

## Imports and settings

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

In [ ]:
# Configuration
LONG_TABLE_STYLE = "style='display:inline-block; max-height:300px; overflow-y:scroll;'"

## Auxiliary functions

In [ ]:
def add_derived_columns(df: pd.DataFrame, mix_rate: bool=False) -> pd.DataFrame:
    df["cpc_before"] = df["cost_before"] / df["clicks_before"]
    df["cpc_after"] = df["cost_after"] / df["clicks_after"]
    df["cpc_delta"] = df["cpc_after"] - df["cpc_before"]
    df["cpc_perc_diff"] = (df["cpc_after"] / df["cpc_before"] - 1) * 100

    return df

def aggregate_data(df: pd.DataFrame, numeric_only: bool=False) -> pd.DataFrame:
    df_agg = pd.DataFrame(df.sum(numeric_only=numeric_only)).T
    df_agg["clicks_before"] = df_agg["clicks_before"].astype(int)
    df_agg["clicks_after"] = df_agg["clicks_after"].astype(int)

    return df_agg

def compute_rate_mix_effects(df: pd.DataFrame) -> pd.DataFrame:
    # Click share columns
    df["click_share_before"] = df["clicks_before"] / df["clicks_before"].sum()
    df["click_share_after"] = df["clicks_after"] / df["clicks_after"].sum()
    df["click_share_delta"] = df["click_share_after"] - df["click_share_before"]

    # Rate-mix effects columns
    df["rate_effect"] = df["click_share_after"] * df["cpc_delta"]
    df["mix_effect"] = df["click_share_delta"] * df["cpc_before"]
    df["total_effect"] = df["rate_effect"] + df["mix_effect"]

    # Percentual rate-mix effects columns
    total_cpc_before = df["cost_before"].sum() / df["clicks_before"].sum()
    df["rate_perc_effect"] = df["rate_effect"] / total_cpc_before * 100
    df["mix_perc_effect"] = df["mix_effect"] / total_cpc_before * 100
    df["total_perc_effect"] = df["rate_perc_effect"] + df["mix_perc_effect"]

    return df

# The data

We read the data set ([download as csv](https://andratx_bellmunt.github.io/portfolio/src/assets/cpc_data.csv)):
  - It contains clicks and costs data for 1,908 digital advertising campaigns
  - Data corresponds to two comparable periods, labelled "before" and "after"

In [ ]:
df_base = pd.read_csv("../assets/cpc_data.csv")
display(
    df_base.style.hide().format(precision=2).set_table_attributes(LONG_TABLE_STYLE)
)

We compute the cost per click (CPC) for before and after and add the delta and the percentual difference between the two:

In [ ]:
df = add_derived_columns(df_base)
display(
    df.style.hide().format(precision=2).set_table_attributes(LONG_TABLE_STYLE)
)

Finally, we aggregate the data to see the global effects:

In [ ]:
df_agg = add_derived_columns(aggregate_data(df, numeric_only=True))
display(df_agg.style.hide().format(precision=2))

# What the stakeholders see and do (and why it does not work)

When looking at the aggregate data the stakeholders see that **the global CPC has increased +18%** and they are alarmed.

Due to operational constraints they cannot realistically take action on more than 100 campaigns, so they need to prioritize.

They proceed as follows:
  - Take the 100 largest campaigns in terms of clicks

  - Sort them according to percentual CPC increase
  
  - Prioritize actions according to that ranking

Let us reproduce this strategy and see what campaigns are tageted:

In [ ]:
print(
    "Top 5 prioritized campaigns according to initial stakeholder's strategy:\n",
    list(
        df
        .sort_values(by="clicks_after", ascending=False)
        .head(100)
        .sort_values(by="cpc_perc_diff", ascending=False)
        .head(5)
        ["campaign_id"]
    )
)

This strategy has two main flaws:
  - **Sorting by percentage CPC increase ignores the base CPC level.** A 20% increase on a $1 click is a +$0.20 change. A 5% increase on a $50 click is $2.50, far more impactful. Percentage change without reference to the baseline is not a measure of impact.

  - **Individual CPC increases do not necessarily imply an aggregate CPC increase.** A set of campaigns can all increase their individual CPCs while the aggregate goes down, and the reverse is equally possible. This happens when the mix of clicks across campaigns changes significantly between the two periods.

## The mix problem: Simpson's paradox

Let us illustrate with a toy example the second of the flaws that we mentioned above:

In [ ]:
df_toy_base = pd.DataFrame(
    columns=["id", "cost_before", "clicks_before", "cost_after", "clicks_after"],
    data=[
        ["A", 1000.00, 100,  200.00, 25],
        ["B",  900.00,  40,  800.00, 40]
    ]
)

df_toy = add_derived_columns(df_toy_base)

df_toy_agg = aggregate_data(df_toy)
df_toy_agg = add_derived_columns(df_toy_agg)

print("Toy example:")
display(df_toy.style.hide().format(precision=2))
print("Aggregated data:")
display(df_toy_agg.style.hide().format(precision=2))

Even in a simple example with only two campaigns we can see how the phenomenon of [Simpson's paradox](https://en.wikipedia.org/wiki/Simpson%27s_paradox) arises:

- Both campaigns **individual CPCs go down**: -20.00% for product A and -11.11% for product B

- However the **aggregated CPC goes up**: +13.36%

This example actually illustrates a very common business situation:

- Campaign A was a large cheap click campaign (CPC=$10) that lost 80% of its volume ($1,000→$200) due to e.g. severe budget cut or dried up audience. Along the way its clicks got cheaper (CPC=$8) because we do not access anymore the more expensive higher quality clicks in that segment.

- Campaign B is a smaller, more expensive campaign (CPC=$22.5) that was optimized and achieved the same number of clicks (40) with a slightly lower budget ($900→$800). This effectively reduced its cost per click (now CPC=$20)

- Both campaigns look fine individually. Yet the aggregate CPC jumps from $13.57 to $15.38 (+13.36%) purely because A's cheap clicks are no longer diluting B's expensive ones.

Hence, the main culprit is that **the mix between the two campaigns has completely changed**. We shall review this in more detail later.

Finally, note that in our real case scenario –where we have 1,900+ campaigns instead of just two– these interactions become much more complex.

# First proposed technical solution (and why it does not work)

In order to navigate the problems we exposed while keeping the stakeholders language (namely "percentual CPC changes") we can borrow a tool from game theory: [Shapley values](https://en.wikipedia.org/wiki/Shapley_value).

- Shapley values measure the contribution of each individual player to a common goal

- To us, each campaign is a player and the common goal is the aggregated CPC percentual change. We want to measure how much each individual campaign contributes to it.

- One property of Shapley values is that individual contributions always add up to the final global result (*efficiency axiom*). E.g. in our toy example above, the sum of the Shapley value of A and the Shapley value of B must be +13.36.

- More in general, in our real data, we shall compute the 1,900 Shapley values and all of them would add up to +18.04.

On paper, this approach works well because we can say to stakeholders "from the global +18.04%, this campaign contributed exactly this much". However, as we will readily see, it fails to capture the real reason behind the CPC surge.

Let us get back to our toy example and compute by the corresponding Shapley values (for only two campaigns the formula is easy to be applied directly):

In [ ]:
print("Shapley values for CPC percentual change:")
print(f"  Campaign A: {round(1/2 * (df_toy_agg['cpc_perc_diff'].iloc[0] + df_toy['cpc_perc_diff'].iloc[0] - df_toy['cpc_perc_diff'].iloc[1]), 2)}%")
print(f"  Campaign B: {round(1/2 * (df_toy_agg['cpc_perc_diff'].iloc[0] + df_toy['cpc_perc_diff'].iloc[1] - df_toy['cpc_perc_diff'].iloc[0]), 2)}%")

This points to B as the main culprit. However, we know from our construction of the example that A is the campaign whose behavior changed most dramatically (it lost 75% of its clicks). B did nothing wrong: its CPC actually decreased.

The reason Shapley misleads us here is that the marginal contribution of each campaign depends not only on its own CPC dynamics (rate effects), but on the volume composition of the coalition it enters (mix effects). A's dramatic volume loss distorts every coalition it participates in, and that distortion gets reflected in B's attributed contribution.

More fundamentally, CPC percentage change conflates two distinct mechanisms: individual CPC changes and click volume mix shifts. Shapley has no way to separate them, so the attribution it produces reflects both entangled together. Attributing percentage CPC change via Shapley is effectively attributing the wrong thing.

# The proper solution

As we have been hinting in previous sections, the contribution of how an individual campaign contributes the global CPC surge depends on the combination of two factors:
- How its CPC changes

- How its share of the total number of clicks shifts

If we denote by $w_i = \frac{\text{clicks}_i}{\text{total\_clicks}}$ the clicks share of campaign $i$, we get the following decomposition:

$$\Delta{\text{CPC}}_{\text{global}} = \sum_i w_i^{\text{after}}\cdot\Delta{\text{CPC}_i} + \sum_i\Delta w_i\cdot\text{CPC}_i^{\text{before}}$$


In the sum, 
- The left term measures **rate effects**: whether the campaign's CPC has improved or worsened, holding clicks share fixed

- The right term measures **mix effects**: whether the clicks mix shifted toward lower or higher CPC campaigns

*Note:* If we want to keep the percentual change narrative, by dividing each term of the sum by $\text{CPC}^{\text{before}}$ we get a percentual equivalent of the rate-mix decomposition.

## Results tables

We now compute the rate-mix effects and display the results

In [ ]:
df = compute_rate_mix_effects(add_derived_columns(df, mix_rate=True))
df_agg = add_derived_columns(aggregate_data(df, numeric_only=True))

### Aggregated results

Let us begin taking a look at the aggregated data to get a general sense of the results we obtained:

In [ ]:
display(
    df_agg.rename_axis("metric").T.rename(columns={0: "value"}).style.format(precision=2)
)

- Just as a double check that our computations are right: as expected, after aggregating, both click shares are 1 and their delta is 0.

- Note also that we get what it was expected in terms of total effects:
  
  - Total effect equals the $0.55 that we got for CPC delta

  - Total percentage effect equals the +18.04% that we got as CPC percentage difference

- Most importantly, this table shows the **the mix effect is much larger than the rate effect**:

  - Of the 55 cents, 49 are explained by mix effects and only 6 by rate effects

  - In percentual terms, of the total +18% increase, 16% comes from mix effects and only 2% from rate effects

These numbers show that the strategy followed by the stakeholders, that mainly focused on rate effects, might be missing relevant data. We shall confirm this in the next section in which we expect the results at individual campaign level.

### Individual campaigns

In [ ]:
def display_mix_rate_effects(df: pd.DataFrame) -> None:
    default_fmt = "{:.2f}"
    overrides = {
        "click_share_before": "{:.6f}",
        "click_share_after": "{:.6f}",
        "click_share_delta": "{:.6f}",
        "rate_effect": "{:.6f}",
        "mix_effect": "{:.6f}",
        "total_effect": "{:.6f}",
        "rate_perc_effect": "{:.6f}",
        "mix_perc_effect": "{:.6f}",
        "total_perc_effect": "{:.6f}",
    }

    fmt = {col: overrides.get(col, default_fmt) for col in df.select_dtypes("float").columns}

    display(
        df
        .sort_values(by="total_effect", ascending=False)
        .style
        .hide()
        .format(fmt)
        .set_table_attributes(LONG_TABLE_STYLE)
    )

print("Results by campaign (sorted by total effect):")
display_mix_rate_effects(df)

A simplified version of this table is exactly the **deliverable that the stakeholders need**: campaigns ranked in order of their effect on the total CPC surge.

In particular we can use it to compare how much of the results are we capturing with the new strategy. Remember:
- Stakeholder's strategy: take top 100 campaigns by number of clicks
- Proposed strategy: take top 100 campaigns ranked by total effect

In [ ]:
df_aux = pd.DataFrame(
    [df.sort_values(by="clicks_after", ascending=False).head(100).sum(numeric_only=True).to_dict(),
    df.sort_values(by="total_effect", ascending=False).head(100).sum(numeric_only=True).to_dict()]
)[["rate_effect", "mix_effect", "total_effect", "rate_perc_effect", "mix_perc_effect", "total_perc_effect"]]

df_aux["new_cpc_delta"] = df_agg["cpc_delta"].iloc[0] - df_aux["total_effect"]
df_aux["new_cpc_perc_diff"] = df_agg["cpc_perc_diff"].iloc[0] - df_aux["total_perc_effect"]

print("Effects captured by Top 100 campaigns:")
display(
    df_aux
    .T
    .reset_index(names="effect")
    .rename(columns={0:"stakeholders_strategy", 1: "rate-mix_effects_strategy"}).style.hide().format(precision=2)
)

- Stakeholder's strategy captures only $0.16 of the the total $0.55 CPC delta (+5.28% of the total +18.04% CPC surge)

- The newly proposed strategy actually *surpasses* the $0.55 CPC delta and accounts for a total effect of $0.66 (+21.87% CPC surge over the registered +18.04%). This of course can happen because there are campaigns with negative total effect.

- Comparing the two strategies (assuming targeted campaigns are moved to 0 total effect):
  - At most stakeholders strategy can bring the CPC delta down from +$0.55 to +$0.39. The rate-mix strategy can bring it down to -$0.11, go beyond compensanting the surge.

  - In percentual terms, the initial strategy can reduce the results from +18.04% to +12.76% CPC surge. With the rate-mix strategy we can prevent the surge all together and even improve the CPCs by -3.83%.

  - In summary, **the new strategy improves results by more than 4x**

## Visualizing the results

### The naive approach

To begin with let us plot the CPC change between before and after. This essentially captures the rate effects. We plot one dot per campaign, with size adjusted (at log scale) by the number of clicks in the "after".

In [ ]:
def plot_cpc_comparison(df: pd.DataFrame) -> None:
    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=df["cpc_before"],
            y=df["cpc_after"],
            mode="markers",
            marker_size=df["clicks_after"].apply(lambda x: 2 * np.log(x)),
            marker_color="#009AD7",
            text=df["campaign_id"],
            name="CPC before/after"
        )
    )

    fig.add_trace(
        go.Scatter(
            x=[0,80],
            y=[0,80],
            mode="lines",
            line=dict(color="red", dash="dash", width=1),
            name="Same CPC before/after"
        )
    )

    fig.update_layout(
        title="CPC comparison",
        width=600,
        height=600,
        xaxis_title="CPC before",
        yaxis_title="CPC after",
        yaxis_scaleanchor="x",
        yaxis_scaleratio=1,
        template="plotly_dark"
    )

    fig.show()

plot_cpc_comparison(df)


To fully replicate the stakeholders strategy let us limit the previous plot to the top 100 campaigns (in terms of clicks). That is, we are taking only the campaigns with the largest $w_i^{\text{after}}$:

In [ ]:
plot_cpc_comparison(df.sort_values(by="clicks_after", ascending=False).head(100))

As we can see all these campaigns are close to the diagonal, so their CPC change is small. Hence, in the rate effects terms $w_i^{\text{after}}\cdot\Delta{\text{CPC}}_i$ we get a large left term and small right term, so these campaigns are not particularly relevant in terms of rate effects. On top of that we are missing the whole mix effects whatsoever, which we already noted are much more relevant in our context. This indeed confirms our concerns about the strategy applied by the stakeholders.

## Rate vs mix effects

In [ ]:
def plot_mix_rate_effect_comparison(df: pd.DataFrame) -> None:
    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=df["rate_effect"],
            y=df["mix_effect"],
            mode="markers",
            marker_size=4,
            marker_color="#009AD7",
            text=df["campaign_id"],
            name="Rate-mix effect"
        )
    )

    for i in range(-1,6):
        fig.add_trace(
            go.Scatter(
                x=[-0.02,0.02],
                y=[0.01 * i + 0.02, 0.01 * i - 0.02],
                mode="lines",
                line=dict(color="red", dash="dash", width=1),
                name=f"Total effect = {0.01 * i}"
            )
        )

    fig.update_layout(
        title="Rate vs Mix effect on CPC change",
        width=600,
        height=600,
        xaxis_title="Rate effect",
        yaxis_title="Mix effect",
        #yaxis_scaleanchor="x",
        #yaxis_scaleratio=1,
        xaxis_range=[-0.021, 0.021],
        showlegend=False,
        template="plotly_dark"
    )

    fig.show()

In [ ]:
plot_mix_rate_effect_comparison(df)

### Rate and mix effects components

In [ ]:
def plot_rate_effect_factors(df: pd.DataFrame) -> None:
    max_abs = np.abs(df["rate_effect"]).max()

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=df["cpc_delta"],
            y=df["click_share_after"],
            mode="markers",
            marker=dict(
                size=5,
                color=df["rate_effect"],
                colorscale="RdBu_r",
                cmin=-max_abs,
                cmax=max_abs,
                colorbar=dict(title="Rate effect")
            ),
            text=df.apply(lambda row: f"{row['campaign_id']} rate effect = {round(row['rate_effect'], 6)}", axis=1)
        )
    )

    fig.update_layout(
        title="Rate effect factors",
        width=600,
        height=600,
        xaxis_title="CPC delta",
        yaxis_title="Click weight after",
        showlegend=False,
        template="plotly_dark"
    )

    fig.show()

In [ ]:
def plot_mix_effect_factors(df: pd.DataFrame) -> None:
    max_abs = np.abs(df["mix_effect"]).max()
    
    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=df["click_share_delta"],
            y=df["cpc_before"],
            mode="markers",
            marker=dict(
                size=5,
                color=df["mix_effect"],
                colorscale="RdBu_r",
                cmin=-max_abs,
                cmax=max_abs,
                colorbar=dict(title="Mix effect")
            ),
            text=df.apply(lambda row: f"{row['campaign_id']} mix effect = {round(row['mix_effect'], 6)}", axis=1)
        )
    )

    fig.update_layout(
        title="Mix effect factors",
        width=600,
        height=600,
        xaxis_title="Weight delta",
        yaxis_title="CPC before",
        showlegend=False,
        template="plotly_dark"
    )

    fig.show()

In [ ]:
plot_rate_effect_factors(df)
plot_mix_effect_factors(df)

# Final comments

- Considering clicks from all campaigns as "the same product" (there are caveats like country or category)
- Note the volumen change from before and after
- Note that for rate-mix decomposition we use CPC change, not percentual change
- We cannot kill campaigns (ofc just keeping the campaigns with negative CPC change would solve the problem)
- Make a comment on robustness
- Make a comment on Shapley values complexity

In [ ]:
# """
# Bootstrap robustness analysis for rate-mix decomposition.

# Given a dataframe of campaigns with clicks/cost before and after,
# this module:
#   1. Computes the exact rate-mix decomposition on the full dataset
#   2. Runs B bootstrap iterations to assess ranking stability
#   3. Returns inclusion frequencies for the top-K campaigns
# """

# import numpy as np
# import pandas as pd


# # ---------------------------------------------------------------------------
# # Rate-mix decomposition
# # ---------------------------------------------------------------------------

# def rate_mix_decomposition(df: pd.DataFrame) -> pd.DataFrame:
#     """
#     Compute rate and mix effects for each campaign.

#     The global CPC delta is decomposed as:
#         delta_CPC_global = sum_i(w_after_i * delta_CPC_i)   [rate effect]
#                          + sum_i(delta_w_i * CPC_before_i)  [mix effect]

#     Parameters
#     ----------
#     df : pd.DataFrame
#         Must contain: clicks_before, cost_before, clicks_after, cost_after.
#         Any other columns are preserved.

#     Returns
#     -------
#     pd.DataFrame with additional columns:
#         cpc_before, cpc_after, delta_cpc,
#         click_weight_before, click_weight_after, delta_weight,
#         rate_effect, mix_effect, total_effect
#     """
#     result = df.copy()

#     total_clicks_before = df["clicks_before"].sum()
#     total_clicks_after  = df["clicks_after"].sum()

#     result["cpc_before"] = df["cost_before"] / df["clicks_before"]
#     result["cpc_after"]  = df["cost_after"]  / df["clicks_after"]
#     result["delta_cpc"]  = result["cpc_after"] - result["cpc_before"]

#     result["click_weight_before"] = df["clicks_before"] / total_clicks_before
#     result["click_weight_after"]  = df["clicks_after"]  / total_clicks_after
#     result["delta_weight"]        = (
#         result["click_weight_after"] - result["click_weight_before"]
#     )

#     result["rate_effect"] = result["click_weight_after"] * result["delta_cpc"]
#     result["mix_effect"]  = result["delta_weight"] * result["cpc_before"]
#     result["total_effect"] = result["rate_effect"] + result["mix_effect"]

#     return result


# def global_cpc_delta(df: pd.DataFrame) -> dict:
#     """
#     Compute global CPC before, after, absolute delta and percentage change.
#     Also returns the rate/mix split of the total effect.
#     """
#     cpc_before = df["cost_before"].sum() / df["clicks_before"].sum()
#     cpc_after  = df["cost_after"].sum()  / df["clicks_after"].sum()
#     delta      = cpc_after - cpc_before
#     pct_change = (cpc_after / cpc_before - 1) * 100

#     decomp = rate_mix_decomposition(df)
#     total_rate = decomp["rate_effect"].sum()
#     total_mix  = decomp["mix_effect"].sum()

#     return {
#         "cpc_before":     cpc_before,
#         "cpc_after":      cpc_after,
#         "delta":          delta,
#         "pct_change":     pct_change,
#         "total_rate_effect": total_rate,
#         "total_mix_effect":  total_mix,
#         "rate_share_pct": total_rate / delta * 100,
#         "mix_share_pct":  total_mix  / delta * 100,
#     }


# # ---------------------------------------------------------------------------
# # Bootstrap robustness
# # ---------------------------------------------------------------------------

# def bootstrap_robustness(
#     df: pd.DataFrame,
#     top_k: int = 100,
#     n_bootstrap: int = 1000,
#     seed: int = 42,
#     id_col: str = "product_id",
# ) -> pd.DataFrame:
#     """
#     Assess ranking robustness of the top-K campaigns via bootstrap resampling.

#     For each bootstrap iteration:
#       - Sample N campaigns with replacement
#       - Recompute the rate-mix decomposition
#       - Record which campaigns appear in the top-K by total effect

#     Parameters
#     ----------
#     df : pd.DataFrame
#         Full campaign dataset. Must contain clicks/cost before/after.
#     top_k : int
#         Number of top campaigns to assess (default 100).
#     n_bootstrap : int
#         Number of bootstrap iterations (default 1000).
#     seed : int
#         Random seed for reproducibility.
#     id_col : str
#         Column name for campaign identifier.

#     Returns
#     -------
#     pd.DataFrame
#         Original decomposition for the full dataset, with an additional
#         column `inclusion_freq` — the fraction of bootstrap samples in
#         which each campaign appeared in the top-K.
#         Sorted by total_effect descending.
#     """
#     rng = np.random.default_rng(seed)
#     n = len(df)

#     # --- Exact decomposition on full dataset --------------------------------
#     full_decomp = rate_mix_decomposition(df)
#     full_decomp = full_decomp.sort_values("total_effect", ascending=False)
#     top_k_ids   = set(full_decomp.head(top_k)[id_col].values)

#     # --- Bootstrap ----------------------------------------------------------
#     # Track how many times each campaign appears in the top-K
#     inclusion_counts = {pid: 0 for pid in df[id_col].values}

#     for _ in range(n_bootstrap):
#         # Sample with replacement
#         sample = df.sample(n=n, replace=True, random_state=rng.integers(1e9))

#         # Recompute decomposition
#         decomp  = rate_mix_decomposition(sample)

#         # Aggregate by product_id in case of duplicates from resampling:
#         # sum clicks and cost, then recompute — this is the correct approach
#         # because a campaign appearing twice means double the volume.
#         agg = (
#             sample
#             .groupby(id_col)[
#                 ["clicks_before", "cost_before", "clicks_after", "cost_after"]
#             ]
#             .sum()
#             .reset_index()
#         )
#         decomp_agg = rate_mix_decomposition(agg)
#         decomp_agg = decomp_agg.sort_values("total_effect", ascending=False)

#         bootstrap_top_k = set(decomp_agg.head(top_k)[id_col].values)

#         for pid in bootstrap_top_k:
#             if pid in inclusion_counts:
#                 inclusion_counts[pid] += 1

#     # --- Attach inclusion frequency -----------------------------------------
#     full_decomp["inclusion_freq"] = (
#         full_decomp[id_col].map(inclusion_counts) / n_bootstrap
#     )

#     return full_decomp.reset_index(drop=True)


# # ---------------------------------------------------------------------------
# # Summary helpers
# # ---------------------------------------------------------------------------

# def robustness_summary(result: pd.DataFrame, top_k: int = 100) -> None:
#     """Print a concise robustness summary for the top-K campaigns."""

#     top = result.head(top_k)
#     thresholds = [0.90, 0.75, 0.50]

#     print("=" * 50)
#     print(f"ROBUSTNESS SUMMARY — TOP {top_k} CAMPAIGNS")
#     print("=" * 50)
#     for t in thresholds:
#         n = (top["inclusion_freq"] >= t).sum()
#         print(f"  Inclusion freq >= {t:.0%}: {n:>4} / {top_k} campaigns")

#     print()
#     print("  Boundary campaigns (rank 80-120):")
#     boundary = result.iloc[79:120][[
#         "product_id", "total_effect", "inclusion_freq"
#     ]]
#     print(boundary.to_string(index=False))
#     print()


# # ---------------------------------------------------------------------------
# # Quick test on toy example
# # ---------------------------------------------------------------------------

# if __name__ == "__main__":

#     df_toy = pd.DataFrame({
#         "product_id":    ["A", "B"],
#         "cost_before":   [1000.0, 900.0],
#         "clicks_before": [100,     40],
#         "cost_after":    [200.0,  800.0],
#         "clicks_after":  [25,      40],
#     })

#     print("--- Global metrics ---")
#     stats = global_cpc_delta(df_toy)
#     for k, v in stats.items():
#         print(f"  {k}: {v:.4f}")

#     print()
#     print("--- Rate-mix decomposition ---")
#     decomp = rate_mix_decomposition(df_toy)
#     print(decomp[[
#         "product_id", "cpc_before", "cpc_after", "delta_cpc",
#         "click_weight_before", "click_weight_after", "delta_weight",
#         "rate_effect", "mix_effect", "total_effect"
#     ]].to_string(index=False))